# A-BLAST MethodsX — v3.2 FINAL rolling-origin rerun
**Use Runtime → Run all.**

This notebook is the locked final rerun package. It mounts Google Drive once, downloads/caches the public datasets, executes the leakage-safe fixed-model rolling-origin evaluation, checkpoints completed seed-level runs directly to Drive, generates Tables 1–6 and Figures 2–4 plus Fig. S1, then stores the final ZIP in Drive.

### Final evaluation design
- chronological 70% / 15% / 15% train / validation / test split;
- 365-day lookback and STL period 365;
- horizons 96, 192, 336, 720;
- five random seeds;
- **five pre-specified rolling forecast origins per horizon**;
- final model fitted once on train+validation and held fixed across test origins;
- STL recomputed from observations strictly before each forecast origin;
- no test-period model refitting.

### Persistence
Results are written to:
`MyDrive/A-BLAST_MethodsX_Rerun_v3_2_FINAL/A-BLAST_MethodsX_RESULTS_v3_2/`

Final archive:
`MyDrive/A-BLAST_MethodsX_Rerun_v3_2_FINAL/A-BLAST_MethodsX_RESULTS_v3_2.zip`

If Colab disconnects, reopen this notebook and choose **Run all** again. Completed dataset × horizon × model × seed checkpoints are skipped. The only unavoidable interaction is Google Drive authorization.


In [ ]:
import base64, io, zipfile, os, pathlib, shutil
_B64 = """..."""
CODE_DIR = pathlib.Path('/content/ABLAST_v3_2_FINAL_CODE')
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
CODE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_B64))) as z:
    z.extractall(CODE_DIR)
os.chdir(CODE_DIR)
print('Code extracted to:', CODE_DIR)
print('Protocol:', pathlib.Path('PROTOCOL_LOCK.json').read_text()[:1200])


In [ ]:
!pip install -q -r requirements.txt
print("Dependencies installed.")


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive', force_remount=False)
DRIVE_BASE = Path('/content/drive/MyDrive/A-BLAST_MethodsX_Rerun_v3_2_FINAL')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
OUT = DRIVE_BASE / 'A-BLAST_MethodsX_RESULTS_v3_2'
FINAL_ZIP = DRIVE_BASE / 'A-BLAST_MethodsX_RESULTS_v3_2.zip'
print('Persistent results folder:', OUT)
print('Persistent final ZIP:', FINAL_ZIP)
print('If this runtime disconnects, run this notebook again; completed checkpoints will be skipped.')


In [ ]:
import subprocess, sys, torch
print("Runtime device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
cmd = [sys.executable, "-u", "run_all.py", "--out", str(OUT)]
print("Running/resuming v3.2 FINAL rolling-origin MethodsX rerun.")
print("Checkpoints are written to Google Drive after each completed dataset × horizon × model × seed run.")
subprocess.run(cmd, check=True)
print("Experiment completed or resumed to completion.")


In [ ]:
from google.colab import files
assert FINAL_ZIP.exists(), f"Final ZIP not found: {FINAL_ZIP}"
print("FINAL RESULT IS SAFE IN GOOGLE DRIVE:")
print(FINAL_ZIP)
print(f"Size: {FINAL_ZIP.stat().st_size/1024/1024:.1f} MB")
print("A browser download will now be offered as an additional copy. Even if it is cancelled, the Drive copy remains safe.")
files.download(str(FINAL_ZIP))
